In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, last, coalesce, lit, avg, max
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Projet_BigData_Sujet1_Final") \
    .getOrCreate()

flights_schema = StructType([
    StructField("flight_id", StringType(), False),
    StructField("flight_date", DateType(), True),
    StructField("airline", StringType(), True),
    StructField("flight_number", StringType(), True),
    StructField("dep_airport", StringType(), True), # Code IATA
    StructField("arr_airport", StringType(), True), # Code IATA
    StructField("scheduled_dep", StringType(), True),
    StructField("actual_dep", StringType(), True),
    StructField("scheduled_arr", StringType(), True),
    StructField("actual_arr", StringType(), True),
    StructField("distance_km", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("ticket_price", DoubleType(), True),
    StructField("cabin_class", StringType(), True),
    StructField("booking_channel", StringType(), True),
    StructField("passenger_id", StringType(), True),
    StructField("loyalty_status", StringType(), True),
    StructField("bags_count", IntegerType(), True),
    StructField("dep_delay_min", DoubleType(), True),
    StructField("arr_delay_min", DoubleType(), True),
    StructField("no_show", IntegerType(), True),
    StructField("weather_dep", StringType(), True),
    StructField("weather_arr", StringType(), True)
])

airports_schema = StructType([
    StructField("airport_code", StringType(), False), # Code APxxx
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("timezone", StringType(), True),
    StructField("traffic_rank", IntegerType(), True)
])

weather_schema = StructType([
    StructField("airport_code", StringType(), False), # Code APxxx
    StructField("date", DateType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("visibility", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("storm_indicator", IntegerType(), True)
])

mapping_schema = StructType([
    StructField("iata_code", StringType(), False),
    StructField("airport_code", StringType(), False)
])

df_flights = spark.read.csv("/content/sample_data/flights_10000.csv", header=True, schema=flights_schema, dateFormat="yyyy-MM-dd")
df_airports = spark.read.csv("/content/sample_data/airports.csv", header=True, schema=airports_schema)
df_weather = spark.read.csv("/content/sample_data/weather_small.csv", header=True, schema=weather_schema, dateFormat="yyyy-MM-dd")
df_mapping = spark.read.csv("/content/sample_data/airport_mapping.csv", header=True, schema=mapping_schema)

vols = df_flights.alias("vols")
count_initial = vols.count()


map_dep = df_mapping.alias("map_dep")
map_arr = df_mapping.alias("map_arr")

vols_mapped = vols.join(
    map_dep, col("vols.dep_airport") == col("map_dep.iata_code"), "left"
).withColumnRenamed("airport_code", "dep_ap_code").drop("iata_code")

vols_mapped = vols_mapped.join(
    map_arr, col("vols.arr_airport") == col("map_arr.iata_code"), "left"
).withColumnRenamed("airport_code", "arr_ap_code").drop("iata_code")

apt_dep = df_airports.alias("apt_dep")
apt_arr = df_airports.alias("apt_arr")

df_airports_joined = vols_mapped.join(
    apt_dep, col("dep_ap_code") == col("apt_dep.airport_code"), "left"
).join(
    apt_arr, col("arr_ap_code") == col("apt_arr.airport_code"), "left"
).select(
    "vols.*", "dep_ap_code", "arr_ap_code",
    col("apt_dep.city").alias("dep_city"), col("apt_dep.country").alias("dep_country"),
    col("apt_arr.city").alias("arr_city"), col("apt_arr.country").alias("arr_country")
)


meteo = df_weather.alias("meteo")
df_base = df_airports_joined.alias("base")

df_explosion = df_base.join(
    meteo, col("base.dep_ap_code") == col("meteo.airport_code"), "left"
)
count_explosion = df_explosion.count()
print(f"--- DÉMONSTRATION EXPLOSION ---")
print(f"Lignes initiales : {count_initial}")
print(f"Lignes après jointure naïve : {count_explosion} (Facteur: {count_explosion/count_initial:.2f}x)\n")

weather_unique = df_weather.dropDuplicates(["airport_code", "date"])


weather_avg = df_weather.groupBy("airport_code", "date").agg(
    avg("wind_speed").alias("wind_speed"),
    avg("visibility").alias("visibility"),
    max("storm_indicator").alias("storm_indicator")
)

condition_meteo = (
    (col("base.dep_ap_code") == weather_avg.airport_code) &
    (col("base.flight_date") == weather_avg.date)
)



df_vols_meteo = df_base.join(weather_avg, condition_meteo, "left")


count_final = df_vols_meteo.count()
print(count_initial)
if count_final > count_initial:
    print(f"❌ ALERTE QUALITÉ : Explosion de cardinalité ! ({count_initial} -> {count_final})")
else:
    print(f"✅ CONTRÔLE RÉUSSI : La cardinalité est préservée ({count_final} lignes).\n")



--- DÉMONSTRATION EXPLOSION ---
Lignes initiales : 10000
Lignes après jointure naïve : 336674 (Facteur: 33.67x)

10000
✅ CONTRÔLE RÉUSSI : La cardinalité est préservée (10000 lignes).



In [5]:
df_flights.printSchema()

root
 |-- flight_id: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- airline: string (nullable = true)
 |-- flight_number: string (nullable = true)
 |-- dep_airport: string (nullable = true)
 |-- arr_airport: string (nullable = true)
 |-- scheduled_dep: string (nullable = true)
 |-- actual_dep: string (nullable = true)
 |-- scheduled_arr: string (nullable = true)
 |-- actual_arr: string (nullable = true)
 |-- distance_km: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- ticket_price: double (nullable = true)
 |-- cabin_class: string (nullable = true)
 |-- booking_channel: string (nullable = true)
 |-- passenger_id: string (nullable = true)
 |-- loyalty_status: string (nullable = true)
 |-- bags_count: integer (nullable = true)
 |-- dep_delay_min: double (nullable = true)
 |-- arr_delay_min: double (nullable = true)
 |-- no_show: integer (nullable = true)
 |-- weather_dep: string (nullable = true)
 |-- weather_arr: string (nullable = true)



In [55]:
df_flights.show()

+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+-----------+---------+------------+-----------+---------------+------------+--------------+----------+-------------+-------------+-------+-----------+-----------+
|flight_id|flight_date|airline|flight_number|dep_airport|arr_airport|scheduled_dep|actual_dep|scheduled_arr|actual_arr|distance_km|   status|ticket_price|cabin_class|booking_channel|passenger_id|loyalty_status|bags_count|dep_delay_min|arr_delay_min|no_show|weather_dep|weather_arr|
+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+-----------+---------+------------+-----------+---------------+------------+--------------+----------+-------------+-------------+-------+-----------+-----------+
|   F00001| 2024-01-08|     KL|       KL6492|        ORY|        LHR|        17:50|     17:50|        20:20|     20:35|       1404|  ON_TIME|      196.85|

In [58]:
df_flights.select("ticket_price").summary().show()

+-------+------------------+
|summary|      ticket_price|
+-------+------------------+
|  count|             10000|
|   mean|225.63555299999936|
| stddev|101.12467838323168|
|    min|             50.01|
|    25%|            138.68|
|    50%|            226.04|
|    75%|            313.75|
|    max|            399.96|
+-------+------------------+



In [6]:
df_airports.printSchema()

root
 |-- airport_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- traffic_rank: integer (nullable = true)



In [7]:
df_weather.printSchema()

root
 |-- airport_code: string (nullable = true)
 |-- date: date (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- storm_indicator: integer (nullable = true)



In [47]:
window_locf = Window.partitionBy("dep_ap_code") \
                    .orderBy("flight_date") \
                    .rowsBetween(Window.unboundedPreceding, 0)

condition_meteo = (
    (col("base.dep_ap_code") == weather_avg.airport_code) &
    (col("base.flight_date") == weather_avg.date)
)
df_vols_meteo = df_base.join(weather_avg, condition_meteo, "left")

df_clean = df_vols_meteo.withColumn(
    "wind_speed_final",
    coalesce(last("wind_speed", True).over(window_locf), lit(0.0))
).withColumn(
    "visibility_final",
    coalesce(last("visibility", True).over(window_locf), lit(10000.0))
)

df_clean.select("flight_id", "dep_ap_code", "flight_date", "wind_speed_final").show(10)

+---------+-----------+-----------+----------------+
|flight_id|dep_ap_code|flight_date|wind_speed_final|
+---------+-----------+-----------+----------------+
|   F00201|      AP001| 2024-01-01|             0.0|
|   F00450|      AP001| 2024-01-01|             0.0|
|   F00729|      AP001| 2024-01-01|             0.0|
|   F01480|      AP001| 2024-01-01|             0.0|
|   F01500|      AP001| 2024-01-01|             0.0|
|   F01820|      AP001| 2024-01-01|             0.0|
|   F02151|      AP001| 2024-01-01|             0.0|
|   F02379|      AP001| 2024-01-01|             0.0|
|   F02561|      AP001| 2024-01-01|             0.0|
|   F03167|      AP001| 2024-01-01|             0.0|
+---------+-----------+-----------+----------------+
only showing top 10 rows


In [48]:
df_clean.count()

10000

In [49]:
df_clean.show()

+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+-----------+-------+------------+-----------+---------------+------------+--------------+----------+-------------+-------------+-------+-----------+-----------+-----------+-----------+--------+-----------+--------+-----------+------------+----+----------+----------+---------------+----------------+----------------+
|flight_id|flight_date|airline|flight_number|dep_airport|arr_airport|scheduled_dep|actual_dep|scheduled_arr|actual_arr|distance_km| status|ticket_price|cabin_class|booking_channel|passenger_id|loyalty_status|bags_count|dep_delay_min|arr_delay_min|no_show|weather_dep|weather_arr|dep_ap_code|arr_ap_code|dep_city|dep_country|arr_city|arr_country|airport_code|date|wind_speed|visibility|storm_indicator|wind_speed_final|visibility_final|
+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+---------

In [50]:
colonnes_a_supprimer = ["airport_code", "date", "wind_speed", "visibility", "precipitation", "storm_indicator"]
df_clean_final  = df_clean.drop(*colonnes_a_supprimer)

In [51]:
cols = ["ticket_price", "wind_speed_final", "visibility_final"]
df_clean_final.select(cols).summary().show()

+-------+------------------+------------------+------------------+
|summary|      ticket_price|  wind_speed_final|  visibility_final|
+-------+------------------+------------------+------------------+
|  count|             10000|             10000|             10000|
|   mean|225.63555299999885| 33.62964575490936| 5762.150576788679|
| stddev|101.12467838323141|23.808118557826603|3257.8925891838057|
|    min|             50.01|               0.0|439.30930263500784|
|    25%|            138.68|12.920027836311725| 2275.041118184612|
|    50%|            226.04|29.860674691935802| 6058.929876907875|
|    75%|            313.75|51.811222159097085|  9056.04134252225|
|    max|            399.96| 78.96081019103727|           10000.0|
+-------+------------------+------------------+------------------+



In [52]:
df_clean_final.show()

+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+-----------+-------+------------+-----------+---------------+------------+--------------+----------+-------------+-------------+-------+-----------+-----------+-----------+-----------+--------+-----------+--------+-----------+----------------+----------------+
|flight_id|flight_date|airline|flight_number|dep_airport|arr_airport|scheduled_dep|actual_dep|scheduled_arr|actual_arr|distance_km| status|ticket_price|cabin_class|booking_channel|passenger_id|loyalty_status|bags_count|dep_delay_min|arr_delay_min|no_show|weather_dep|weather_arr|dep_ap_code|arr_ap_code|dep_city|dep_country|arr_city|arr_country|wind_speed_final|visibility_final|
+---------+-----------+-------+-------------+-----------+-----------+-------------+----------+-------------+----------+-----------+-------+------------+-----------+---------------+------------+--------------+----------+-------------+-------

In [53]:
df_clean_final.explain(True)

== Parsed Logical Plan ==
Project [flight_id#4457, flight_date#4458, airline#4459, flight_number#4460, dep_airport#4461, arr_airport#4462, scheduled_dep#4463, actual_dep#4464, scheduled_arr#4465, actual_arr#4466, distance_km#4467, status#4468, ticket_price#4469, cabin_class#4470, booking_channel#4471, passenger_id#4472, loyalty_status#4473, bags_count#4474, dep_delay_min#4475, arr_delay_min#4476, no_show#4477, weather_dep#4478, weather_arr#4479, dep_ap_code#4522, arr_ap_code#4526, ... 6 more fields]
+- Project [flight_id#4457, flight_date#4458, airline#4459, flight_number#4460, dep_airport#4461, arr_airport#4462, scheduled_dep#4463, actual_dep#4464, scheduled_arr#4465, actual_arr#4466, distance_km#4467, status#4468, ticket_price#4469, cabin_class#4470, booking_channel#4471, passenger_id#4472, loyalty_status#4473, bags_count#4474, dep_delay_min#4475, arr_delay_min#4476, no_show#4477, weather_dep#4478, weather_arr#4479, dep_ap_code#4522, arr_ap_code#4526, ... 11 more fields]
   +- Projec